In [1]:
REPO <- "https://raw.githubusercontent.com/vicleme/PI3-FFV/main/"   # endereco Raw do grupo (ficha, secao 1)

In [6]:
REPO <- "https://raw.githubusercontent.com/vicleme/PI3-FFV/main/"   # endereco Raw do grupo (ficha, secao 1)
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor07.R")   # funcoes do curso, ate a Aula 06
download.file(paste0(REPO, "estrutura/codigo/config.R"), "config.R")         # traz o config.R
source("config.R")                                                           # cfg: as decisoes do grupo
cp     <- read.csv(paste0(REPO, "estrutura/banco-de-dados/corpus.csv"),      # o corpus congelado na Aula 05
                   stringsAsFactors = FALSE, fileEncoding = "UTF-8")
docs   <- setNames(cp$texto, cp$id)                                          # vetor nomeado id -> texto
origem <- readRDS(gzcon(url(paste0(REPO, "estrutura/banco-de-dados/origem.rds"))))   # de que verbete veio cada documento
ix     <- montar(docs, cfg)                                                  # indice, TF-IDF e BM25
estado()                                                                     # a ultima linha: o que eu preciso ver

<!-- estado-R:inicio -->
## Estado do R ao fim da sessão
*gerado por `anexar_estado()` em 2026-11-25 23:08 · R version 4.6.1 (2026-06-24) · motor07 v1 2026-10-15 · não edite à mão*

### Objetos

| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `a` | character | 1 | qrels_consenso.csv |
| `arqs` | character | 1 | qrels_consenso.csv |
| `cfg$minimo` | numeric | 1 | 200 |
| `cfg$limpar` | logical | 1 | TRUE |
| `cfg$acentos` | character | 1 | translit |
| `cfg$stopwords` | character | 203 | a, ao, aos, aquela, aquelas, aquele, ... |
| `cfg$k1` | numeric | 1 | 1.2 |
| `cfg$b` | numeric | 1 | 0.75 |
| `cfg$limiar` | numeric | 1 | 2 |
| `cfg$juizes` | character | 1 | consenso |
| `cfg$prf_k` | numeric | 1 | 0 |
| `cfg$rocchio` | numeric, nomeado | 3 | a 1, b 0.75, g 0.15 |
| `cfg$grupo` | character | 1 | FFV |
| `consultas` | character, nomeado | 10 | q01 fundacao santos ano, q02 praias santos quantidade, q03 maior porto america latina, q04 polo industrial, q05 tamanho guaru

In [5]:
arqs <- c("qrels_consenso.csv")                                           # o gabarito de consenso do grupo
for (a in arqs) download.file(paste0(REPO, "estrutura/banco-de-dados/", a), a)   # traz o qrels para o Colab
qrels <- ler_qrels(arqs, juiz = cfg$juizes)                                 # le o gabarito
nec   <- read.csv(paste0(REPO, "estrutura/banco-de-dados/necessidades.csv"),     # as necessidades, com a coluna conjunto
                  stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")
consultas <- setNames(nec$texto_consulta, nec$consulta)                     # q01 -> o texto digitado
R_q  <- sapply(qrels, function(g) sum(g >= cfg$limiar))                     # relevantes por consulta, com o limiar do grupo
usar <- names(qrels)[R_q > 0]                                               # as que entram na avaliacao
names(qrels)[R_q == 0]                                                      # AVISO: as que ficam fora (R = 0)

character(0)

In [7]:
lsa <- function(ix, k) {                                          # k: quantas dimensoes guardar
  s <- svd(ix$w)                                                  # W = U Sigma V'
  U <- s$u[, 1:k, drop = FALSE]; rownames(U) <- ix$vocab          # termos x k
  D <- s$v[, 1:k, drop = FALSE] %*% diag(s$d[1:k], nrow = k)      # documentos x k: V_k Sigma_k
  rownames(D) <- colnames(ix$w)                                   # os ids dos documentos
  list(U = U, S = s$d[1:k], D = D)                                # o modelo
}
ranking_lsa <- function(consulta, ix, cfg, modelo) {              # modelo: o que lsa() devolve
  q0 <- vetor_consulta(preparar(consulta, cfg), ix$vocab, ix$idf_tfidf)  # texto -> um peso por termo
  qk <- t(modelo$U) %*% q0                                        # dobra a consulta: k coordenadas
  s  <- apply(modelo$D, 1, function(d) cosseno(qk, d))            # um cosseno por LINHA (documento)
  sort(s, decreasing = TRUE)                                      # do maior ao menor
}

In [8]:
dim(ix$w); min(dim(ix$w))                         # termos x documentos; o k maximo e o menor dos dois
m20 <- lsa(ix, 20)                                # LSA de trabalho, k = 20
s <- svd(ix$w)                                    # a SVD inteira, so para a energia
round(sum(m20$S^2) / sum(s$d^2), 3)               # fracao da energia guardada com k = 20

[1] 1518   63

[1] 63

[1] 0.486

In [9]:
q  <- "q10"                                          # uma consulta de DESENVOLVIMENTO (troquem)
rl <- ranking_lsa(consultas[[q]], ix, cfg, m20)      # o LSA: o TEXTO da consulta, dobrado em 20 coordenadas
rb <- ranking_bm25(consultas[[q]], ix, cfg)          # o lexico, com as decisoes do grupo
rbind(bm25 = names(rb)[1:5], lsa = names(rl)[1:5])   # os dois top-5 lado a lado

bm25,Guaruja_p19,Guaruja_p10,Santos_p16,Guaruja_p4,Guaruja_p18
lsa,Guaruja_p19,Cubatao_p8,Guaruja_p2,Cubatao_p9,Cubatao_p4


In [10]:
so_lsa <- names(rl)[1:5][!names(rl)[1:5] %in% names(rb)[1:5]]   # os que SO o LSA trouxe
so_lsa                                                          # os ids
docs[so_lsa]                                                    # os textos: leiam

[1] "Cubatao_p8" "Guaruja_p2" "Cubatao_p9" "Cubatao_p4"

Cubatao_p8 
                                                                                            "Em 12 de agosto de 1833, a Regência Trina Permanente sancionou a Lei 24, que separou um terreno da Fazenda de Cubatão para ali fundar uma povoação. Contudo, o povoado não se desenvolveu e o município não chegou a ser constituído. Pela Lei Regencial 167 de 1 de março de 1841, o Povoado de Cubatão é incorporado à cidade de Santos. A emancipação de Cubatão ocorreria 108 anos depois, em 1949, e é este o motivo de estarem inscritas as duas datas no Brasão da cidade." 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     Guaruja_p2 
    "Atualmente, a cidade de Guarujá é conhecida como a Pérola do Atlântico, devido às suas belas praias e belezas naturais. Muito procurada pelos turistas na alta temporada, a cidade conta com praias urbanizadas e algumas selvagens, acessíveis apenas por trilhas. Além do litoral, Guarujá oferece construções históricas e trilhas de ecoturismo. Outra atração local é a pesca artesanal, que pode ser vista e praticada em diversas praias do município ao longo de sua orla. Anualmente, a cidade recebe mais de 1,5 milhão de turistas de diversas partes do país." 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     Cubatao_p9 
"Com o desenvolvimento da região, foi construída, em 1947, a Via Anchieta, chamada de orgulho da engenharia rodoviária nacional, a mais moderna das rodovias brasileiras na época; com o crescimento, o fortalecimento comercial e industrial, o bairro começou a perder suas características agrícolas e começou a sonhar com o futuro. Organizou-se, em 1948, um grupo de trabalho para lutar pela elevação de Cubatão à categoria de município, que ficou conhecido como Os Emancipadores. A emancipação política e autonomia de Cubatão aconteceria em 9 de abril de 1949." 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     Cubatao_p4 
                                                                                                                                                                                                                                                      "É ponto pacífico entre os estudiosos que a região do atual município de Cubatão foi ocupada, há pelo menos 7 000 anos, pelo chamado Homem do Sambaqui: eram grupos seminômades, que tinham sua vida cotidiana ligada ao mangue, onde tiravam sua subsistência, marcada por siris, conchas bivalves, berbigões e peixes."

In [11]:
sapply(so_lsa, function(d)                                          # para cada um que so o LSA trouxe...
  length(intersect(preparar(consultas[[q]], cfg), ix$tokens[[d]])))  # ...quantas palavras da consulta ele tem
qrels[[q]][so_lsa]                                                  # e o grau que o grupo deu (0, 1 ou 2)

Cubatao_p8 Guaruja_p2 Cubatao_p9 Cubatao_p4 
         1          1          0          0

Cubatao_p8 Guaruja_p2 Cubatao_p9 Cubatao_p4 
         0          0          0          0

In [12]:
so_bm25 <- names(rb)[1:5][!names(rb)[1:5] %in% names(rl)[1:5]]   # os que SO o BM25 trouxe
docs[so_bm25]                                                    # os textos: leiam
qrels[[q]][so_bm25]                                              # o grau de cada um

Guaruja_p10 
"Em 1911, a companhia é adquirida pelo empresário norte-americano Percival Farquhar, passando a se denominar Companhia Guarujá. O novo Grand Hôtel de la Plage foi reinaugurado em 1912, tendo sido um marco para o turismo de luxo no Brasil. O sucesso do hotel e a reputação de Guarujá como destino de verão da classe alta paulistana levam a um contínuo desenvolvimento da vila durante a primeira metade do século XX. Neste mesmo hotel, em 1932, morreria o aeronauta e inventor brasileiro Alberto Santos Dumont, onde havia se hospedado após sérios problemas de saúde." 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           Santos_p16 
                                                                                                                                                                             "O intenso processo de conurbação atualmente em curso na região vem criando uma metrópole cujo centro está em Santos e atinge vários municípios, como São Vicente, Praia Grande e Guarujá. A Região Metropolitana da Baixada Santista foi criada mediante Lei Complementar Estadual 815, em 30 de julho de 1996, tornando-se a primeira região metropolitana brasileira sem status de capital estadual." 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           Guaruja_p4 
                                                                                                                                        "Conforme explica Eduardo Navarro em seu Dicionário de Tupi Antigo, Guarujá é derivado do termo tupi agûarausá, que designa um tipo de caranguejo, o guaruçá. A Ilha de Santo Amaro surge em sua atual forma no final da Era Glacial, entre 20 000 e 123 000 anos atrás, quando o Canal de Bertioga e o Estuário de Santos são abertos com a contínua elevação do nível do Oceano Atlântico e criam a atual ilha, separando-a do continente." 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          Guaruja_p18 
                                                                 "O turismo, sazonal, e os ganhos advindos do mercado imobiliário-turístico movimentam parte significativa da economia de Guarujá. A outra parte, relevante e não sazonal, advém do porto, na margem esquerda do Porto de Santos, e atividades afins, tais como transporte. Devido a sua proximidade com Cubatão, maior distrito industrial do país, e portos, existe também interesse pela ocupação industrial na região, iniciada em 1976 pela Dow Química, ainda hoje a única grande indústria a ocupar a região."

Guaruja_p10  Santos_p16  Guaruja_p4 Guaruja_p18 
          0           0           0           0

In [13]:
rrf <- function(..., c = 60) {                         # ...: quantos rankings vierem
  rks <- list(...)                                     # os rankings, numa lista
  ids <- unique(unlist(lapply(rks, names)))            # todos os documentos que aparecem
  s <- sapply(ids, function(d)                         # para cada documento...
    sum(sapply(rks, function(r) {                      # ...soma sobre os rankings
      p <- which(names(r) == d)                        # a posicao dele neste ranking
      if (length(p)) 1 / (c + p) else 0 })))           # ausente: nao soma nada
  sort(s, decreasing = TRUE)                           # do maior ao menor
}

In [14]:
ranking_modelo <- function(txt, modelo, ix, cfg, m, c = 60) {    # m: o que lsa() devolve
  rb <- ranking_bm25(txt, ix, cfg)                               # o lexico
  if (modelo == "bm25") return(rb)                               # so ele, e a funcao termina aqui
  rl <- ranking_lsa(txt, ix, cfg, m)                             # o LSA
  if (modelo == "lsa") rl else rrf(rb, rl, c = c)                # so ele, ou a fusao RRF
}
ap_modelo <- function(q, modelo, m, consultas, qrels, ix, cfg)   # o AP de um modelo numa consulta
  metricas(ranking_modelo(consultas[[q]], modelo, ix, cfg, m), qrels[[q]], cfg$limiar)[["AP"]]

In [15]:
ks <- c(5, 10, 20, 40)                                               # os k candidatos do grupo
ks <- ks[ks <= min(dim(ix$w))]                                       # so os possiveis
ms <- setNames(lapply(ks, function(k) lsa(ix, k)), paste0("k", ks))  # um modelo LSA por k
names(ms)                                                            # quais ficaram

[1] "k5"  "k10" "k20" "k40"

In [16]:
dev   <- intersect(usar, nec$consulta[nec$conjunto == "dev"])          # as consultas de desenvolvimento
teste <- intersect(usar, nec$consulta[nec$conjunto == "teste"])        # so no fim, uma vez
tab_dev <- sapply(dev, function(q) c(                                  # para cada consulta dev, o AP de...
  bm25 = ap_modelo(q, "bm25", NULL, consultas, qrels, ix, cfg),        # ...BM25 (nao usa LSA: m = NULL)
  lsa  = sapply(ms, function(m) ap_modelo(q, "lsa", m, consultas, qrels, ix, cfg)),       # ...LSA, cada k
  hib  = sapply(ms, function(m) ap_modelo(q, "hibrido", m, consultas, qrels, ix, cfg))))  # ...hibrido, cada k
round(tab_dev, 3); round(rowMeans(tab_dev), 3)                         # por consulta, e o MAP de cada linha

,q02,q03,q05,q06,q07,q09,q10
bm25,0.500,1.000,0.077,1.00,0.367,0.500,1.0
lsa.k5,0.020,0.290,0.026,0.25,0.077,0.267,0.5
lsa.k10,0.034,0.833,0.059,1.00,0.174,0.361,1.0
lsa.k20,0.067,1.000,0.053,1.00,0.156,0.417,1.0
lsa.k40,0.333,1.000,0.042,1.00,0.200,0.417,1.0
hib.k5,0.071,0.833,0.056,1.00,0.208,0.500,1.0
hib.k10,0.100,1.000,0.077,1.00,0.243,0.500,1.0
hib.k20,0.200,1.000,0.083,1.00,0.267,0.500,1.0
hib.k40,0.333,1.000,0.056,1.00,0.333,0.500,1.0


bm25  lsa.k5 lsa.k10 lsa.k20 lsa.k40  hib.k5 hib.k10 hib.k20 hib.k40 
  0.635   0.204   0.494   0.527   0.570   0.524   0.560   0.579   0.603

In [17]:
colnames(tab_dev)[tab_dev["lsa.k40", ] < tab_dev["bm25", ]]   # consultas em que o LSA (k = 40) perdeu do BM25
colnames(tab_dev)[tab_dev["hib.k40", ] < tab_dev["bm25", ]]   # e em que o hibrido (k = 40) perdeu

[1] "q02" "q05" "q07" "q09"

[1] "q02" "q05" "q07"

In [18]:
q   <- "q02"                                                  # a consulta que vamos ler
rel <- names(qrels[[q]])[qrels[[q]] >= cfg$limiar]            # o(s) relevante(s) do gabarito
rb  <- ranking_bm25(consultas[[q]], ix, cfg)                  # o lexico
rh  <- ranking_modelo(consultas[[q]], "hibrido", ix, cfg, ms[["k40"]])   # o hibrido, k = 40
rel; which(names(rb) == rel); which(names(rh) == rel)         # o id e a posicao em cada um
rbind(bm25 = names(rb)[1:3], hib = names(rh)[1:3])            # os tres primeiros de cada
docs[rel]                                                     # o texto do relevante: leiam

[1] "Santos_p14"

[1] 2

[1] 3

bm25,Santos_p11,Santos_p14,Guaruja_p2
hib,Santos_p11,Guaruja_p2,Santos_p14


Santos_p14 
"A orla santista é composta por 6 praias em oito quilômetros de extensão. Santos possui clima tropical litorâneo úmido. Os verões são quentes e úmidos, enquanto os invernos têm como característica temperaturas mais amenas e menor incidência de chuvas."

In [19]:
rl <- ranking_lsa(consultas[[q]], ix, cfg, ms[["k40"]])             # o LSA sozinho, k = 40
which(names(rl) == "Santos_p14"); which(names(rl) == "Guaruja_p2")  # a posicao de cada um no LSA

[1] 3

[1] 1

In [20]:
escolha <- "bm25"                                                    # a decisao do grupo
tab_teste <- sapply(teste, function(q) c(                            # para cada consulta de teste...
  bm25      = ap_modelo(q, "bm25", NULL, consultas, qrels, ix, cfg),     # ...a referencia
  escolhido = ap_modelo(q, escolha, NULL, consultas, qrels, ix, cfg)))   # ...e a escolha (m = NULL)
cbind(round(tab_teste, 3), MAP = round(rowMeans(tab_teste), 3))      # AP por consulta e o MAP, em tabela

,q01,q04,q08,MAP
bm25,0.056,0.333,0.062,0.15
escolhido,0.056,0.333,0.062,0.15


In [21]:
source("config.R")                                                     # recarrega com os campos novos
cfg[c("modelo", "lsa_k", "rrf_c")]                                     # os tres campos, conferir com a frase
ranking_motor <- function(txt, ix, cfg)                                # o ranking do motor, com a decisao
  ranking_modelo(txt, cfg$modelo, ix, cfg, lsa(ix, cfg$lsa_k), c = cfg$rrf_c)
round(head(ranking_motor(consultas[["q07"]], ix, cfg), 5), 3)          # top-5 da q07 (dev)

$modelo
[1] "bm25"

$lsa_k
[1] 40

$rrf_c
[1] 60

Santos_p15 Cubatao_p1 Cubatao_p8 Cubatao_p4 Cubatao_p9 
     4.251      3.295      3.273      2.996      2.992

In [22]:
q   <- "q05"                                                  # a consulta do ganho
rel <- names(qrels[[q]])[qrels[[q]] >= cfg$limiar]            # o relevante do gabarito
rb  <- ranking_bm25(consultas[[q]], ix, cfg)                  # o lexico
rh  <- ranking_modelo(consultas[[q]], "hibrido", ix, cfg, ms[["k20"]])   # o hibrido, k = 20
rel; which(names(rb) == rel); which(names(rh) == rel)         # o id e a posicao em cada um
rbind(bm25 = names(rb)[10:13], hib = names(rh)[10:13])        # as posicoes 10 a 13 de cada
docs[rel]                                                     # o texto do relevante: leiam

[1] "Guaruja_p1"

[1] 13

[1] 12

bm25,Santos_p16,Guaruja_p16,Guaruja_p4,Guaruja_p1
hib,Santos_p8,Guaruja_p16,Guaruja_p1,Santos_p3


Guaruja_p1 
"Guarujá é um município brasileiro do estado de São Paulo. Localiza-se na Região Metropolitana da Baixada Santista. A população, segundo dados do IBGE de 2022, é de 287 634 habitantes. Possui uma área de 144,794 km², o que resulta numa densidade demográfica de 1 986,50 habitante por quilômetro quadrado. Geograficamente, situa-se na Ilha de Santo Amaro, terceira maior ilha do litoral paulista. O município é formado pela sede e pelo distrito de Vicente de Carvalho."

In [23]:
anexar_estado("00_FICHA_PROJETO.md")            # o estado do R vai para a ficha
anexar_estado("aula07_parteD_consolidado.md")   # e para o consolidado do grupo

Estado do R anexado a 00_FICHA_PROJETO.md (71 linhas de objetos).

Estado do R anexado a aula07_parteD_consolidado.md (71 linhas de objetos).

